# Binary classifier: MMGBSA energies &rarr; Myc/Max DNA binding

This notebook reproduces the **binary classification** path of the canonical NN pipeline (`ML_models/ML_NN/run_model.py`, `ML_models/ML_NN/hyperopt.py`, and the data-prep logic in `ML_models/ML_RF/Data/torch_prep_kfold.py`) as a single end-to-end notebook with intermediate outputs and explanations.

### The task

Given 8 MMGBSA-derived energy features (`VDWAALS`, `EEL`, `EGB`, `ESURF`, `HB Energy`, `Hydrophobic Energy`, `Pi-Pi Energy`, `Delta_Entropy`) measured from 20-replicate MD trajectories of each Myc/Max&ndash;DNA complex, **predict whether the 36-bp DNA sequence is a binder or a non-binder**. Threshold: `Log Intensity = 7.7` on the gcPBM normalised intensity.

### Inputs (must exist in the working directory)

| File | Produced by | Shape |
|---|---|---|
| `exp_data_all.csv` | `scripts/process_gcPBM.ipynb` | 168 rows &times; `{sequence, bind_avg, binding_type, improving}` |
| `rawdat.csv` | `scripts/AMBER_MMGBSA.ipynb` | 272 160 rows &times; `{sequence, run, 8 energy terms}` |

### Outputs (written to the working directory)

`bin_trn_final.csv`, `bin_tst_preprocess.csv`, `bin_trn_{rep}_{fold}.csv`, `bin_val_{rep}_{fold}.csv`, `bin_train_stats.csv`, `bin_tst_final.csv`, `Model/nn_fold_{rep}_{fold}_bin.pth`, `predictions_bin_*.csv`, `predictions_test_bin*.csv`, `final_metrics_bin_*.csv`.

### How this differs from the canonical scripts

* Uses **fixed hyperparameters** from an earlier Hyperopt run (no Spark cluster needed).
* Runs `5 x 3 = 15` K-fold splits instead of the canonical `5 x 5 = 25` to keep the notebook quick.
* Uses an intuitive `binding_binary` label (`1 = binder`) instead of the canonical `improving` column (`1 = non-binder`). Accuracy and MCC are unaffected by this flip &mdash; see section 3.
* Records per-epoch train + val loss and **plots learning curves**. Canonical scripts only log loss every 10% of epochs.

Sections below are ordered top-to-bottom to run cleanly with **Run All**.


## 1. Setup &mdash; imports, seed, configuration


In [ ]:
# Standard library
import os
import csv
import logging
from collections import defaultdict
from typing import Any, List, Tuple

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch
import torch as T

# scikit-learn pieces actually used below
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    matthews_corrcoef,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


In [84]:
random_state = 42
test_percentage = 0.20
np.random.seed(42)

In [ ]:
# --- Configuration: data-prep + model hyperparameters ----------------------
# These values are the result of an earlier hyperopt run on the canonical
# pipeline (see ML_models/ML_NN/hyperopt.py); we use them as fixed
# hyperparameters here so the tutorial runs end-to-end without a Spark cluster.

# ---------- data-prep ----------
KEEP_LAST_PERCENT = 60       # fraction of frames retained per sequence
NAVG              = 60       # rows averaged per chunk (per sequence)

# ---------- model architecture ----------
HIDDEN_LAYERS        = 3
HIDDEN_SIZE          = 8
DROPOUT_INPUT_OUTPUT = 0.13
DROPOUT_HIDDEN       = 0.33

# ---------- optimiser / training ----------
LEARNING_RATE = 0.005
WEIGHT_DECAY  = 7.69e-7
MAX_EPOCHS    = 5000
PATIENCE      = 50           # early-stopping patience (epochs)

# ---------- cross-validation ----------
KFOLD       = 5              # k folds per repeat
NUM_REPEATS = 3              # repeated K-fold splits

device = T.device("cuda" if T.cuda.is_available() else "cpu")
device


**Where these values came from.** The numbers above are the output of an earlier Hyperopt+SparkTrials sweep over the canonical pipeline (see `ML_models/ML_NN/hyperopt.py` and the `best_hyperparams_bin.json` file `run_hyperopt.sh` writes). They are pinned here so the notebook runs end-to-end without needing a Spark cluster.

Hyperopt searches over:

* `KEEP_LAST_PERCENT` &mdash; fraction of MD frames kept per sequence (see the note in section 5 for what this *actually* does in the current pipeline).
* `NAVG` &mdash; number of frames averaged into each model-input row (per sequence).
* `HIDDEN_LAYERS`, `HIDDEN_SIZE` &mdash; MLP architecture.
* `LEARNING_RATE`, `WEIGHT_DECAY`, `DROPOUT_INPUT_OUTPUT`, `DROPOUT_HIDDEN` &mdash; optimiser + regularisation.

`KFOLD`, `NUM_REPEATS`, and `MAX_EPOCHS` / `PATIENCE` are fixed (not searched).


## 2. Load gcPBM labels + MMGBSA features

We load two tables and inner-join them on `sequence`:

* `exp_data_all.csv` &mdash; one row per sequence with the experimental labels (`bind_avg`, `binding_type`, `improving`). We keep `binding_type` here because the next section derives the binary label from it.
* `rawdat.csv` &mdash; one row per MD frame, with the 8 MMGBSA-derived energy terms. The `run` column (replicate index, 1&ndash;20) is dropped via `usecols=` so it doesn't get standardised later as if it were a feature.

After the merge each sequence is associated with ~1620 feature rows, one per (frame &times; replicate). We then shuffle the whole table with a fixed seed so the upcoming sequence-grouped split doesn't get systematic ordering.


In [87]:
# torch_prep_kfold.py
# 
###########################################################################
# 1) INITIAL SPLIT
###########################################################################
id_col = 'sequence'
label_col = 'binding_type'
df1 = pd.read_csv('exp_data_all.csv')
ref_data = df1[[id_col, label_col]].copy()

print(ref_data)

## experimental training
usecols = ['sequence','VDWAALS','EEL','EGB','ESURF','HB Energy','Hydrophobic Energy','Pi-Pi Energy','Delta_Entropy']

df2 = pd.read_csv('rawdat.csv', usecols=usecols)
feature_data = df2.copy()

print(feature_data)

                                 sequence  binding_type
0    GAGGAAGCAGCCCTCGCCCCTGTCGGTGGAAAGAAG             0
1    GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG             0
2    ATCTGATCAAAACAACGAATTCCAAAACAAAGTAAT             0
3    CCAATATTCCTTTGTGAGACCCTCCACAAATGCTAA             0
4    GAGGACGCGAACCGGCACGCTGCGCCTTTAAGGAGT             0
..                                    ...           ...
163  ACATAGGGACGGGGCCATGCGGTGGGCGGGTGGAAC             2
164  GAAAACCAGCGAGACCGCATGGTCTCACTTATAAGT             2
165  CGCGGAGACCCGAAGCACGTGGTATCCATACTAGTT             2
166  GCCCCCGACCCCGCGCACGCGGCCCCGCCCCGCGCG             2
167  ATTAGCCAAACTAAACACGTGTATTGATTTTAGATG             2

[168 rows x 2 columns]
                                    sequence  VDWAALS       EEL       EGB  \
0       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -252.110 -1886.830  1841.253   
1       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -238.510 -1881.424  1835.847   
2       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -246.721 -1895.687  1851.589

In [88]:
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print(df_merged.head(), df_merged.shape)

                               sequence  VDWAALS       EEL       EGB   ESURF  \
0  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -252.110 -1886.830  1841.253 -36.482   
1  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -238.510 -1881.424  1835.847 -36.023   
2  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -246.721 -1895.687  1851.589 -35.802   
3  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -235.671 -1857.573  1814.002 -34.799   
4  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC -230.214 -1897.268  1847.934 -34.391   

   HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  binding_type  
0  -1.940432         -165.447020 -1.655191e-03     -26.046553             2  
1  -2.003962         -155.422935 -4.708262e-02     -24.150637             2  
2  -2.269901         -142.386371 -5.901517e-29     -24.329875             2  
3  -2.838678         -147.918585 -3.236084e-07     -23.615145             2  
4  -2.810414         -151.012478 -1.784784e-05     -23.698348             2   (272160, 10)


In [89]:
df_merged = df_merged.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


## 3. Derive the binary label

The reference table `exp_data_all.csv` already ships with a binary column called **`improving`**, computed in `scripts/process_gcPBM.ipynb` as

```python
exp['improving'] = exp['bind_avg'].apply(lambda x: 0 if x >= 7.7 else 1)
```

i.e. `improving = 0` for **binders** (`Log Intensity >= 7.7`) and `improving = 1` for **non-binders**. The name reflects the lab's mutational-engineering framing &mdash; "this sequence still has room to improve" = currently a non-binder &mdash; but it's the *opposite* of the conventional ML labeling where "positive class = the thing we want to detect."

For pedagogical clarity, this tutorial instead derives `binding_binary` from the three-class `binding_type` column:

* `binding_binary = 0` &rarr; non-binder (`binding_type == 0`, i.e. `Log Intensity <= 7.7`)
* `binding_binary = 1` &rarr; binder (`binding_type` in `{1, 2}`, i.e. medium or strong binding)

So `binding_binary == 1 - improving` (modulo the threshold-boundary edge case).

**Effect on metrics:** Accuracy and MCC are **mathematically invariant under joint label flipping** &mdash; flipping `0 <-> 1` on both `y_true` and `y_pred` leaves both numbers unchanged. Since the canonical `run_model.py` only reports ACC and MCC for `bin`, this tutorial's headline numbers match the paper exactly. The confusion-matrix cell *positions* swap (the "true binder, predicted binder" cell ends up in a different corner), but the counts and the story they tell are equivalent. F1 would differ &mdash; this tutorial doesn't compute it, but `process_results.ipynb` does.


In [ ]:
df_merged["binding_binary"] = df_merged["binding_type"].apply(lambda x: 0 if x == 0 else 1)
label_col = 'binding_binary'
df_merged = df_merged.drop(columns=["binding_type"])

# Sequence-level class balance: each sequence has a single label, regardless of how
# many MD frames are associated with it, so we collapse by sequence first.
per_seq = df_merged.groupby("sequence")["binding_binary"].first()
print("Sequence-level class balance:")
print(f"  Class 0 (Non-binding):  {(per_seq == 0).sum():3d}")
print(f"  Class 1 (Binding)    :  {(per_seq == 1).sum():3d}")
print(f"  Total sequences      :  {len(per_seq):3d}")


### 3.1 Sanity check: feature distributions by class

Before any modelling, plot the eight MMGBSA features split by `binding_binary`. If the two classes overlap completely on every feature, no model will help us; if some features separate the classes well, the model has something to latch onto.


In [ ]:
feature_names = ['VDWAALS', 'EEL', 'EGB', 'ESURF',
                 'HB Energy', 'Hydrophobic Energy', 'Pi-Pi Energy', 'Delta_Entropy']

fig, axes = plt.subplots(2, 4, figsize=(15, 6.5), sharey=False)
for ax, feat in zip(axes.flat, feature_names):
    ax.hist(df_merged.loc[df_merged["binding_binary"] == 0, feat],
            bins=50, alpha=0.55, color="tab:blue",   label="Non-binder (0)")
    ax.hist(df_merged.loc[df_merged["binding_binary"] == 1, feat],
            bins=50, alpha=0.55, color="tab:orange", label="Binder (1)")
    ax.set_title(feat, fontsize=10)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Per-frame MMGBSA feature distributions, split by class", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()


## 4. Train / test split (grouped by sequence)

We hold out roughly 1/7 (~14%) of the **sequences** as the test set, matching the canonical pipeline (`run_hyperopt.sh` / `run_ML.sh` &rarr; `torch_prep_kfold.py --initial_split`). Two properties to enforce:

1. **No sequence appears in both train and test.** Each sequence has ~1620 correlated MD frames; if those frames straddle train and test, the model can essentially memorise sequence-specific noise &mdash; classic data leakage. `StratifiedGroupKFold` with `groups=sequence` guarantees a hard separation.
2. **Both classes are represented in train and test.** With only 168 sequences this matters &mdash; a plain `GroupKFold` could easily push all binders into one half. `StratifiedGroupKFold` balances the binary label *while* keeping sequences grouped.

`n_splits=7` is the canonical choice: 1/7 &asymp; 14% test, 6/7 &asymp; 86% train. We take just the first split.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

y_for_split = df_merged[label_col]
groups = df_merged[id_col]

gkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=random_state)
train_idx, test_idx = next(gkf.split(df_merged, y_for_split, groups=groups))

df_train = df_merged.iloc[train_idx].copy()
df_test  = df_merged.iloc[test_idx].copy()

# --- Sanity checks ---------------------------------------------------------
trn_seqs = set(df_train["sequence"].unique())
tst_seqs = set(df_test["sequence"].unique())

print(f"Train: {df_train.shape[0]:,} rows / {len(trn_seqs):3d} sequences")
print(f"Test : {df_test.shape[0]:,}  rows / {len(tst_seqs):3d} sequences")
print(f"Overlap                : {len(trn_seqs & tst_seqs)} sequences  (must be 0)")
print(f"Total sequences covered: {len(trn_seqs | tst_seqs)}        (expected 168)")

# Class balance per split (at the sequence level)
trn_balance = df_train.groupby('sequence')[label_col].first().value_counts().reindex([0, 1], fill_value=0)
tst_balance = df_test .groupby('sequence')[label_col].first().value_counts().reindex([0, 1], fill_value=0)
print(f"\nClass balance (sequence-level):")
print(f"  Train  0/1:  {trn_balance[0]:3d} / {trn_balance[1]:3d}")
print(f"  Test   0/1:  {tst_balance[0]:3d} / {tst_balance[1]:3d}")


In [92]:
print(df_train.columns)

Index(['sequence', 'VDWAALS', 'EEL', 'EGB', 'ESURF', 'HB Energy',
       'Hydrophobic Energy', 'Pi-Pi Energy', 'Delta_Entropy',
       'binding_binary'],
      dtype='object')


In [93]:
print(df_train)

                                    sequence  VDWAALS       EEL       EGB  \
0       CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT -207.264 -1958.570  1908.590   
1       TCCTAAACAGGAAGCCATGAGGTGAGCAGAGACACT -229.177 -1917.175  1870.986   
2       TTAGAAAAATAGTTTAAAATCTAGAGTTAATTAACC -197.506 -1830.441  1787.958   
3       CCAGCTCTCCACCGCCGCGTGCGCCTGCAGACGCTC -207.553 -1891.794  1845.707   
4       CCCCCAGCGCTCCGGCACGCGCCGGGAGACCTCCGG -201.484 -1923.635  1874.505   
...                                      ...      ...       ...       ...   
272155  TCTCCCCTTCCTCTCCGCGTGGCGGGCGCGGGTGCG -212.840 -1935.569  1883.328   
272156  TGTGGAGCAAGGGAGACAGAAGCTCATTGGCTAGAG -196.231 -1914.625  1866.175   
272157  AGAGGTAGGGTTAGGCGCGTGCCGCGAGAACAGAGT -186.779 -1891.556  1842.579   
272158  CCAGCCTGGACCGCCCCTGTGGGCTCCACTCCCCTC -199.291 -1903.864  1856.077   
272159  CTCAGGGCAGTTGGGCACGTGGGGTCCGGCTGGTTG -230.435 -1981.724  1926.780   

         ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy 

In [94]:
# save

train_file = f"bin_trn_final.csv"
test_file  = f"bin_tst_preprocess.csv"
df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

## 5. Train-set preprocessing

The raw training frame (~232 k rows after the split) gets transformed through three steps before it reaches the model. All three are exactly what `torch_prep_kfold.py --process train` does in the canonical pipeline; we just split them across cells so you can inspect intermediate state.

1. **`keep_last_percent` filtering** &mdash; retain a fraction of rows per sequence.
2. **Per-sequence chunk averaging** &mdash; average `NAVG=60` frames into one model-input row.
3. **Standardisation** &mdash; fit a `(mean, std)` table on the averaged train set, then z-score.

The helpers in the next cell are direct ports of the functions in `ML_models/ML_RF/Data/torch_prep_kfold.py`.


In [ ]:
## helper functions

def keep_last_n_percent(df: pd.DataFrame, seq_col: str, keep_percent: float) -> pd.DataFrame:
    """
    Retain only the last keep_percent fraction of rows in each sequence group.
    If 'run' column exists, sort by it first.
    """
    if keep_percent <= 0 or keep_percent >= 100:
        return df
    if "run" in df.columns:
        df_sorted = df.sort_values([seq_col, "run"], kind="mergesort")
    else:
        df_sorted = df.copy()
    group_sizes = df_sorted.groupby(seq_col)[seq_col].transform("size")
    cumcount = df_sorted.groupby(seq_col).cumcount()
    n_keep = (group_sizes * (keep_percent / 100.0)).astype(int)
    n_keep = n_keep.mask(n_keep < 1, 1)  # ensure at least 1 row if fraction>0
    mask = cumcount >= (group_sizes - n_keep)
    return df_sorted[mask].reset_index(drop=True)

def average_features_for_sequence(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """
    Shuffle the sequence's rows, chunk into size navg, and average numeric features.
    Label = label from the first row of each chunk.
    """
    results = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in [id_col, label_col]]
    df_shuffled = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    n_chunks = len(df_shuffled) // navg
    if n_chunks < 1:
        return pd.DataFrame(columns=df.columns)
    for i in range(n_chunks):
        chunk = df_shuffled.iloc[i * navg : (i + 1) * navg]
        row_dict = {col: chunk[col].mean() for col in feature_cols}
        row_dict[label_col] = chunk[label_col].iloc[0]
        row_dict[id_col] = chunk[id_col].iloc[0]
        results.append(row_dict)
    df_out = pd.DataFrame(results)
    col_order = [id_col] + sorted([c for c in df_out.columns if c != id_col])
    return df_out[col_order]

def average_features_for_mutants(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """
    Apply the above averaging function per sequence group, then concat.
    """
    all_chunks = []
    for seq, group in df.groupby(id_col):
        chunk_df = average_features_for_sequence(group, navg, id_col, label_col, random_state)
        all_chunks.append(chunk_df)
    if not all_chunks:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(all_chunks, ignore_index=True)

def compute_mean_std(
    df: pd.DataFrame,
    model_type: str,
    id_col: str,
    label_col: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute mean/std for numeric columns (excluding ID and label).
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_col in numeric_cols:
        numeric_cols.remove(id_col)
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)
    means = [(c, df[c].mean()) for c in numeric_cols]
    stds  = [(c, df[c].std())  for c in numeric_cols]
    return (
        pd.DataFrame(means, columns=["colname", "mean"]),
        pd.DataFrame(stds, columns=["colname", "std"])
    )

def save_mean_std(mean_df: pd.DataFrame, std_df: pd.DataFrame, filename: str) -> None:
    merged = pd.merge(mean_df, std_df, on="colname")
    merged.to_csv(filename, index=False)

def load_mean_std(filename: str) -> pd.DataFrame:
    """
    Load a previously-saved (mean, std) table and validate its schema.
    Mirrors the canonical torch_prep_kfold.py: refuses to return a frame
    that's missing any of the required columns.
    """
    df_stats = pd.read_csv(filename)
    needed_cols = {"colname", "mean", "std"}
    missing = needed_cols - set(df_stats.columns)
    if missing:
        raise ValueError(
            f"Mean/Std file {filename} is missing required columns: {sorted(missing)}. "
            f"Found: {df_stats.columns.tolist()}"
        )
    return df_stats

def apply_standardization(
    df: pd.DataFrame,
    stats_df: pd.DataFrame,
    model_type: str,
    id_col: str,
    label_col: str
) -> pd.DataFrame:
    df_std = df.copy()
    means = dict(zip(stats_df["colname"], stats_df["mean"]))
    stds  = dict(zip(stats_df["colname"], stats_df["std"]))
    for col in df_std.columns:
        if col in [id_col, label_col]:
            continue
        if col in means and col in stds:
            mu  = means[col]
            sigma = stds[col]
            if sigma == 0 or np.isnan(sigma):
                df_std[col] = df_std[col] - mu
            else:
                df_std[col] = (df_std[col] - mu) / sigma
    return df_std


### 5.1 `keep_last_percent` filtering &mdash; a known quirk to be aware of

The helper above is *intended* to keep the **last X% of frames per sequence**, i.e. the equilibrated tail of each MD trajectory, by sorting each sequence's rows by the `run` column first.

In this pipeline `run` was dropped when we set `usecols` while reading `rawdat.csv`, so the `if "run" in df.columns:` branch never fires here. The canonical pipeline (`ML_models/ML_NN/torch_prep_kfold.py`) drops `run` at the same point too, so its behavior is identical.

What `keep_last_n_percent` therefore *actually* does in both this tutorial and the canonical code is: keep a deterministic X% per sequence in whatever order the rows currently appear &mdash; which, after the upstream shuffle, is effectively a random subsample. The hyperparameter `keep_last_percent` is still meaningful (it controls the fraction of frames retained), but it does **not** select the equilibrated tail.

We keep the tutorial bug-for-bug consistent with the canonical scripts here so that the figures in the paper remain reproducible. This is recorded as a known issue in `CLAUDE.md` &rarr; *Open questions / risks*.


In [96]:
df_train = keep_last_n_percent(df_train, id_col, KEEP_LAST_PERCENT)


In [97]:
print(df_train.shape)

(139968, 10)


### 5.2 Per-sequence chunk averaging

Each sequence has ~1600 MD-derived rows even after step 5.1. Feeding those directly to a small MLP wastes signal &mdash; adjacent frames are highly correlated. Instead we **shuffle within each sequence and average chunks of `NAVG=60` rows** into a single model-input row, carrying the label (which is constant per sequence) through unchanged.

Two effects:

* **Noise reduction.** Each averaged row now reflects an ensemble over `NAVG` independent (shuffled) MD frames, smoothing single-frame fluctuations in the energy terms.
* **Effective sample-size control.** With `NAVG=60` and ~970 frames/sequence (after 60% keep), each sequence yields ~16 averaged rows. So the optimiser sees ~`144 sequences x 16 ~= 2,300` training rows &mdash; enough to fit an MLP without overfitting on the raw 144 sequences, while still small enough that a single epoch costs almost nothing.

`NAVG` is jointly tuned with the model hyperparameters in the canonical Hyperopt sweep, which is part of why it might surprise you to see a value like 60 rather than 1.


In [98]:
# Average numeric features in chunks of size --navg
df_train_avg = average_features_for_mutants(
    df_train,
    navg=NAVG,
    id_col=id_col,
    label_col=label_col,
    random_state=random_state)

### 5.3 Standardisation

We compute `(mean, std)` per feature on the **averaged training set**, save them to `bin_train_stats.csv`, then z-score the train features in place. Later (in section 7) we apply the *same* statistics to the test set &mdash; we never refit on test, since that would let test-set information leak into the standardiser.

The `sequence` ID and the label column are explicitly skipped so they pass through untouched.


In [99]:
# Compute and save mean/std
mean_df, std_df = compute_mean_std(df_train_avg, 'bin', id_col, label_col)
stats_file = f"bin_train_stats.csv"
save_mean_std(mean_df, std_df, stats_file)


In [ ]:
# Apply standardization
df_train_std = apply_standardization(
    df_train_avg,
    load_mean_std(stats_file),
    'bin',
    id_col,
    label_col
)

# Sanity: standardised features on TRAIN should have mean ~ 0 and std ~ 1
feature_cols = [c for c in df_train_std.columns if c not in [id_col, label_col]]
print("Post-standardisation stats (train set, feature columns only):")
print(df_train_std[feature_cols].describe().T[["mean", "std", "min", "max"]].round(3).to_string())


In [101]:
# Shuffle once more
df_train_std = df_train_std.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


## 6. Repeated K-fold splits on the training set

For each of `NUM_REPEATS=3` repeats we generate a fresh `KFOLD=5`-fold split &mdash; **15 (train, val) pairs** in total, each saved to disk as `bin_trn_{rep}_{fold}.csv` / `bin_val_{rep}_{fold}.csv`.

Two design choices to call out:

* **`StratifiedGroupKFold` again.** Same reason as the outer split: keep all rows of one sequence in the same fold, *and* keep both classes represented in train + val.
* **Repeats give error bars.** A single K-fold gives 5 point estimates (one per fold); 3 repeats give 15. With only 168 sequences, single-fold variance is high, so the repeated split lets us report `mean ± std` for ACC and MCC in section 9.

Canonical uses `5 x 5 = 25` splits; we use `5 x 3 = 15` so the tutorial finishes in a couple of minutes on CPU.


In [102]:
# Repeated K-fold

from sklearn.model_selection import StratifiedGroupKFold
y_for_split = df_train_std[label_col]
groups      = df_train_std[id_col]



In [103]:

for repeat_idx in range(NUM_REPEATS):
    # (Optional) offset seed if you want different splits each repeat
    repeat_seed = random_state + 100 * repeat_idx

    kf = StratifiedGroupKFold(n_splits=KFOLD, shuffle=True, random_state=repeat_seed)
    split_iter = kf.split(df_train_std, y_for_split, groups)

    fold_counter = 0
    for trn_idx, val_idx in split_iter:
        df_fold_trn = df_train_std.iloc[trn_idx].copy()
        df_fold_val = df_train_std.iloc[val_idx].copy()

        col_order = [id_col] + [c for c in df_fold_trn.columns if c != id_col]
        df_fold_trn = df_fold_trn[col_order]
        df_fold_val = df_fold_val[col_order]

        fold_train_csv = f"bin_trn_{repeat_idx}_{fold_counter}.csv"
        fold_val_csv   = f"bin_val_{repeat_idx}_{fold_counter}.csv"
        df_fold_trn.to_csv(fold_train_csv, index=False)
        df_fold_val.to_csv(fold_val_csv, index=False)

        fold_counter += 1


In [ ]:
# Inspect each saved fold: row counts, sequence counts, and class balance.
# This confirms StratifiedGroupKFold honored both grouping AND stratification.
print(f"{'rep':>3s} {'fold':>4s}   {'trn rows':>9s} {'val rows':>9s}   "
      f"{'trn seqs':>9s} {'val seqs':>9s}   {'trn 0/1':>9s} {'val 0/1':>9s}")
for rep in range(NUM_REPEATS):
    for fold in range(KFOLD):
        trn = pd.read_csv(f"bin_trn_{rep}_{fold}.csv")
        val = pd.read_csv(f"bin_val_{rep}_{fold}.csv")

        trn_bal = trn[label_col].value_counts().reindex([0, 1], fill_value=0).tolist()
        val_bal = val[label_col].value_counts().reindex([0, 1], fill_value=0).tolist()

        print(f"{rep:3d} {fold:4d}   "
              f"{len(trn):9,d} {len(val):9,d}   "
              f"{trn[id_col].nunique():9d} {val[id_col].nunique():9d}   "
              f"{trn_bal[0]:4d}/{trn_bal[1]:<4d} {val_bal[0]:4d}/{val_bal[1]:<4d}")


## 7. Test-set preprocessing

We re-run the same three steps on the held-out test set: `keep_last_percent` filtering, `NAVG=60` chunk averaging, and standardisation. The crucial point is that **standardisation uses the training mean/std** (loaded from `bin_train_stats.csv`), not statistics re-fit on the test set &mdash; otherwise the standardiser would leak test-set distribution information.

After this section we save `bin_tst_final.csv`, ready for the evaluation loop in section 10.


In [104]:
print(df_test.shape)
# 1 sequence + 8 features + 1 label

(38880, 10)


In [105]:
df_test = keep_last_n_percent(df_test, id_col, KEEP_LAST_PERCENT)
df_test_avg = average_features_for_mutants(
    df_test,
    navg=NAVG,
    id_col=id_col,
    label_col=label_col,
    random_state=random_state)

In [106]:
df_test_std = apply_standardization(
    df_test_avg,
    load_mean_std(stats_file),
    'bin',
    id_col,
    label_col
)

In [107]:
#shuffle
df_test_std = df_test_std.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

final_test_csv = f"bin_tst_final.csv"
df_test_std.to_csv(final_test_csv, index=False)

## 8. Model definitions

The next few cells define the building blocks &mdash; load helper, the `Net` MLP, loss-function selector, early stopper, train/eval routines, CSV writer &mdash; in the same shape as `ML_models/ML_NN/run_model.py`.

**Architecture** (matches `Net` in the canonical script):

```
[8 features]
   |
   Linear(HIDDEN_SIZE=8)  -> ReLU  -> Dropout(DROPOUT_INPUT_OUTPUT=0.13)
   Linear(8)              -> ReLU  -> Dropout(DROPOUT_HIDDEN=0.33)
   Linear(8)              -> ReLU  -> Dropout(DROPOUT_HIDDEN=0.33)
   Linear(1)                                          -> logit
```

Despite `HIDDEN_LAYERS=3` and `HIDDEN_SIZE=8` looking tiny, those values came out of the Hyperopt sweep &mdash; the model is intentionally lean because we only have ~150 training sequences.

**Loss**: `BCEWithLogitsLoss` (binary cross-entropy on the raw logit, numerically stable; sigmoid is folded into the loss).

**Optimiser**: Adam with `lr=5e-3`, weight decay `7.7e-7`.

**Early stopping**: monitor validation loss, stop after `PATIENCE=50` epochs without improvement.


In [110]:
'''
Modes:
------
1) mode=0 (training):
   - For each fraction in --scramble_fractions,
     - For each repeat_idx in [0..num_repeats-1] and fold_idx in [0..kfold-1]:
         * Load the CSV files:
             {prefix}_{model_type}_scrFRAC_trn_{repeat_idx}_{fold_idx}.csv
             {prefix}_{model_type}_scrFRAC_val_{repeat_idx}_{fold_idx}.csv
         * Instantiate the MLP with arguments:
             --hidden_size, --hidden_layers, --dropout_input_output, --dropout_hidden
         * Train the network (with optional early stopping);
           save best model checkpoint to:
             nn_fold_{repeat_idx}_{fold_idx}_{model_type}_scrFRAC.pth
         * Evaluate on the validation split → collect metrics & row-level predictions.
'''

'\nModes:\n------\n1) mode=0 (training):\n   - For each fraction in --scramble_fractions,\n     - For each repeat_idx in [0..num_repeats-1] and fold_idx in [0..kfold-1]:\n         * Load the CSV files:\n             {prefix}_{model_type}_scrFRAC_trn_{repeat_idx}_{fold_idx}.csv\n             {prefix}_{model_type}_scrFRAC_val_{repeat_idx}_{fold_idx}.csv\n         * Instantiate the MLP with arguments:\n             --hidden_size, --hidden_layers, --dropout_input_output, --dropout_hidden\n         * Train the network (with optional early stopping);\n           save best model checkpoint to:\n             nn_fold_{repeat_idx}_{fold_idx}_{model_type}_scrFRAC.pth\n         * Evaluate on the validation split → collect metrics & row-level predictions.\n'

In [111]:
###############################################################################
# Utility Functions
###############################################################################
def fmt_float(x: float) -> str:
    """
    Format a float with 4 digits after the decimal.
    If `x` is NaN or not a float → return the string 'NaN'.
    """
    return f"{x:.4f}" if isinstance(x, float) and not np.isnan(x) else "NaN"


def majority_vote(values: List[int]) -> int:
    """
    Return the most common integer in *values*.
    Ties are broken by `pandas.Series.mode()`, which returns the
    first encountered mode.
    """
    return int(pd.Series(values).mode()[0])

In [112]:
###############################################################################
# Data Loading
###############################################################################
def load_csv_data(csv_file: str, id_col, label_col) -> Tuple[T.Tensor, T.Tensor, List[Any]]:
    """
    Read a CSV file and convert to PyTorch tensors.
    *   ID   column → kept as Python list for aggregation.
    * target column → 1-D float32 tensor (with extra dim so shape = [N,1]).
    * feature cols → float32 tensor  (shape = [N, #features])
    """
    if not os.path.isfile(csv_file):
        raise FileNotFoundError(f"Data file not found: {csv_file}")

    df = pd.read_csv(csv_file, header=0)
    id_col, label_col = id_col, label_col

    if id_col not in df.columns or label_col not in df.columns:
        raise ValueError(
            f"CSV {csv_file} must contain '{id_col}' and '{label_col}' columns."
        )

    # treat every other column as numeric feature
    feature_cols = [c for c in df.columns if c not in [id_col, label_col]]

    ids      = df[id_col].tolist()
    features = T.tensor(df[feature_cols].values.astype(np.float32))
    targets  = T.tensor(df[label_col].values.astype(np.float32)).unsqueeze(1)
    return features, targets, ids

In [ ]:
###############################################################################
# Neural Network Definition
###############################################################################
class Net(T.nn.Module):
    """
    Simple fully-connected network mirroring ML_models/ML_NN/run_model.py:

        [input] -> Linear(hidden_size) -> ReLU -> Dropout(io)
                -> (hidden_layers - 1) x [Linear(hidden_size) -> ReLU -> Dropout(hid)]
                -> Linear(out_dim)

    Output layer width:
      * regression / binary classification -> 1 neuron
      * multi-class                       -> `num_classes` neurons (logits)
    """
    def __init__(self, input_dim: int,
                 hidden_size: int,
                 num_classes: int,
                 hidden_layers: int,
                 dropout_input_output: float,
                 dropout_hidden: float,
                 model_type: str = 'bin'):
        super().__init__()

        # output dim depends on task: only mclass needs >1 output unit
        out_dim = num_classes if model_type == "mclass" else 1
        self.model_type = model_type

        # ---------------- network architecture --------------------------- #
        layers = []
        # first layer (input -> hidden)
        layers.append(T.nn.Linear(input_dim, hidden_size))
        # additional hidden layers
        for _ in range(hidden_layers - 1):
            layers.append(T.nn.Linear(hidden_size, hidden_size))
        # final output
        layers.append(T.nn.Linear(hidden_size, out_dim))
        self.layers = T.nn.ModuleList(layers)

        # activation / dropout
        self.act            = T.nn.ReLU()
        self.dropout_io     = T.nn.Dropout(dropout_input_output)
        self.dropout_hidden = T.nn.Dropout(dropout_hidden)

        self._init_weights()   # Xavier init

    def _init_weights(self):
        """Xavier-uniform for weights, zeros for biases."""
        for layer in self.layers:
            T.nn.init.xavier_uniform_(layer.weight)
            T.nn.init.zeros_(layer.bias)

    def forward(self, x: T.Tensor) -> T.Tensor:
        """
        Forward pass with dropout after first & hidden layers.
        """
        x = self.dropout_io(self.act(self.layers[0](x)))
        for layer in self.layers[1:-1]:
            x = self.dropout_hidden(self.act(layer(x)))
        return self.layers[-1](x)   # logits / regression value


In [ ]:
###############################################################################
# Loss Function 
###############################################################################
def get_loss_function(model_type='bin'):
    """Return the appropriate criterion given task type."""
    if model_type == "reg":
        return T.nn.MSELoss()
    elif model_type == "bin":
        return T.nn.BCEWithLogitsLoss()
    elif model_type == "mclass":
        return T.nn.CrossEntropyLoss()
    else:
        raise ValueError(f"Unknown model_type '{model_type}'")


In [115]:
###############################################################################
# Early-Stopping Helper
###############################################################################
class EarlyStopper:
    """
    Monitor a validation metric (`min` or `max`) and stop training if it
    hasn't improved for `patience` epochs. Stores the best model weights.
    """
    def __init__(self, patience=20, mode="min"):
        self.patience   = patience
        self.mode       = mode
        self.best_value = None
        self.counter    = 0
        self.should_stop = False
        self.best_model_state = None

    def check(self, current_value, model_state_dict):
        """
        Update internal state; set `should_stop` if patience exceeded.
        """
        if self.best_value is None:
            # first observation
            self.best_value = current_value
            self.best_model_state = model_state_dict
            return

        improved = (
            current_value < self.best_value if self.mode == "min"
            else current_value > self.best_value
        )

        if improved:
            self.best_value = current_value
            self.best_model_state = model_state_dict
            self.counter = 0           # reset patience
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


In [ ]:
def train_model(
    net: "Net",
    trn_feat: T.Tensor,
    trn_tgt: T.Tensor,
    val_feat: T.Tensor,
    val_tgt: T.Tensor,
    lrn_rate: float,
    wt_decay: float,
    max_epochs: int,
    use_early_stopping: bool = True,
    patience: int = 20,
    model_type: str = 'bin'
):
    """
    Full-batch gradient descent training loop with optional early stopping.

    Two differences from the canonical run_model.py training loop:
      1. We compute val-loss EVERY epoch (not just when early-stopping is on),
         so we can plot learning curves regardless.
      2. We record `(epoch, train, val)` losses on `net.loss_history` so the
         outer fold-loop can stash them for plotting in section 9 below.
    """

    # loss criterion (BCEWithLogitsLoss for binary by default)
    criterion = get_loss_function(model_type=model_type)

    # optimizer
    optimizer = T.optim.Adam(
        net.parameters(),
        lr=lrn_rate,
        weight_decay=wt_decay
    )

    # CrossEntropyLoss expects class-index LongTensor of shape [N], not [N, 1] float
    if model_type == "mclass":
        trn_tgt = trn_tgt.view(-1).long()
        val_tgt = val_tgt.view(-1).long()

    # learning-curve buffers, attached to the network so the outer loop can read them back
    net.loss_history = {"epoch": [], "train": [], "val": []}

    # early stopping
    early_stopper = (
        EarlyStopper(patience=patience, mode="min")
        if use_early_stopping else None
    )

    log_every = max(1, max_epochs // 10)

    for ep in range(max_epochs):
        # ---- one gradient step on TRAIN ------------------------------- #
        optimizer.zero_grad()
        preds = net(trn_feat)
        train_loss = criterion(preds, trn_tgt)
        train_loss.backward()
        optimizer.step()

        # ---- compute VAL loss for this epoch (also feeds early stopping) #
        net.eval()
        with T.no_grad():
            val_loss = criterion(net(val_feat), val_tgt).item()
        net.train()

        # record both losses
        net.loss_history["epoch"].append(ep + 1)
        net.loss_history["train"].append(train_loss.item())
        net.loss_history["val"].append(val_loss)

        # periodic logging
        if (ep + 1) % log_every == 0:
            logging.info(
                f"[Epoch {ep+1}/{max_epochs}] "
                f"train-loss={train_loss.item():.4f} val-loss={val_loss:.4f}"
            )

        # ---- early stopping ------------------------------------------- #
        if early_stopper:
            early_stopper.check(val_loss, net.state_dict())
            if early_stopper.should_stop:
                logging.info(
                    f"Early stopping at epoch {ep+1} "
                    f"(best val-loss={early_stopper.best_value:.4f})"
                )
                net.load_state_dict(early_stopper.best_model_state)
                break


In [117]:
###############################################################################
# CSV Helper
###############################################################################
def save_predictions(predictions, filename: str):
    """
    Write a CSV with three columns: *Label*, *Predicted*, *True*.
    `predictions` is an iterable of tuples (label, pred, tgt).
    """
    with open(filename, "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["Label", "Predicted", "True"])
        wr.writerows(predictions)

In [118]:
###############################################################################
# Evaluation / Prediction
###############################################################################
def evaluate_model(net: Net,
                   features: T.Tensor,
                   targets: T.Tensor,
                   ids: List[Any],
                   ):
    """
    Forward-prop through the network and compute metrics.  
    Multiple rows may share the same ID (e.g., snapshots of same complex).
    Rows → aggregated to per-ID predictions:

    * Regression  → average of predictions.
    * Classification → majority vote (or summed logits for mclass).
    """
    net.eval()
    with T.no_grad():
        out = net(features)

    logits = out.detach().cpu().numpy().flatten()  # raw scores
    tgt_np = targets.detach().cpu().numpy().flatten()
    row_preds = (logits > 0.0).astype(int)

    aggregator = defaultdict(lambda: {"preds": [], "tgt": []})
    row_level_data = []

    for i, uid in enumerate(ids):
        aggregator[uid]["preds"].append(int(row_preds[i]))
        aggregator[uid]["tgt"].append(int(tgt_np[i]))
        row_level_data.append((uid, int(row_preds[i]), int(tgt_np[i])))

    agg_preds = [majority_vote(d["preds"]) for d in aggregator.values()]
    agg_tgts  = [majority_vote(d["tgt"])  for d in aggregator.values()]

    mcc, acc = float('nan'), float('nan')
    if len(set(agg_tgts)) > 1:
        mcc = matthews_corrcoef(agg_tgts, agg_preds)
        acc = accuracy_score(agg_tgts,  agg_preds)

    return (np.nan, np.nan, np.nan, mcc, acc, row_level_data)


## 9. Training: repeated K-fold cross-validation

For each of the 15 `(rep, fold)` pairs prepared in section 6 we:

1. Load the train + validation CSVs into tensors.
2. Instantiate a fresh `Net` (Xavier-uniform init).
3. Train with `train_model()` (Adam, full-batch GD, BCE loss, early stopping on val loss).
4. Save the fold checkpoint to `Model/nn_fold_{rep}_{fold}_bin.pth`.
5. Evaluate on the held-out *validation* fold and stash predictions for later aggregation.
6. Stash the per-epoch `(train_loss, val_loss)` learning curve so we can plot all 15 of them at the end.

The cell below takes a couple of minutes on CPU.


In [ ]:
## TRAINING

model_dir = "Model/"   # Directory for saving/loading models.
data_dir  = "."
os.makedirs(model_dir, exist_ok=True)

all_metrics = []
all_predictions = []
all_loss_histories = []   # one entry per (rep, fold) -> consumed by the learning-curve plot below

# ------------------- iterate repeats / folds -------------------- #
for repeat_idx in range(NUM_REPEATS):
    for fold_idx in range(KFOLD):
        # -------- locate CSVs produced by the prep section above ----- #
        trn_file = os.path.join(
            data_dir,
            f"bin_trn_{repeat_idx}_{fold_idx}.csv"
        )
        val_file = os.path.join(
            data_dir,
            f"bin_val_{repeat_idx}_{fold_idx}.csv"
        )

        # ------------------- load data --------------------------- #
        X_trn, y_trn, ids_trn = load_csv_data(trn_file, id_col, label_col)
        X_val, y_val, ids_val = load_csv_data(val_file, id_col, label_col)

        # ------------------- build & train ----------------------- #
        net = Net(
            input_dim=X_trn.shape[1],
            model_type="bin",
            num_classes=1,                # ignored for binary
            hidden_size=HIDDEN_SIZE,
            hidden_layers=HIDDEN_LAYERS,
            dropout_input_output=DROPOUT_INPUT_OUTPUT,
            dropout_hidden=DROPOUT_HIDDEN,
        )

        train_model(
            net, X_trn, y_trn, X_val, y_val,
            lrn_rate=LEARNING_RATE,
            wt_decay=WEIGHT_DECAY,
            max_epochs=MAX_EPOCHS,
            use_early_stopping=True,
            patience=PATIENCE,
        )

        # stash this fold's learning curves for plotting later
        all_loss_histories.append({
            "rep":   repeat_idx,
            "fold":  fold_idx,
            **net.loss_history,
        })

        # save model checkpoint
        model_path = os.path.join(
            model_dir,
            f"nn_fold_{repeat_idx}_{fold_idx}_bin.pth"
        )
        T.save(net.state_dict(), model_path)

        # ------------------- validation metrics ------------------ #
        mse, r2, pear, mcc, acc, fold_preds = evaluate_model(
            net, X_val, y_val, ids_val
        )
        all_metrics.append(
            {"MSE": mse, "R2": r2, "Pear": pear, "MCC": mcc, "Accuracy": acc}
        )
        all_predictions.extend(fold_preds)

        # save per-fold prediction CSV
        fold_csv = f"predictions_bin_rep{repeat_idx}_fold{fold_idx}.csv"
        save_predictions(fold_preds, fold_csv)


In [120]:
# ------------------ aggregate metrics across folds -------------- #
if all_metrics:
    avg_metrics = {}
    for key in all_metrics[0]:
        vals = [m[key] for m in all_metrics if not np.isnan(m[key])]
        avg_metrics[key] = float(np.mean(vals)) if vals else float('nan')

    for k,v in avg_metrics.items():
        logging.info(f"  {k} = {fmt_float(v)}")

    # write metrics CSV
    with open(f"final_metrics_bin_trn_.csv",
                "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["MSE","R2","Pear","MCC","Accuracy"])
        w.writeheader()
        w.writerow({k: fmt_float(v) for k,v in avg_metrics.items()})

In [122]:
# ------------------ aggregate predictions by ID ----------------- #
if all_predictions:
    aggregator = defaultdict(lambda: {"preds": [], "tgt": []})
    for uid, pred, tgt in all_predictions:
        aggregator[uid]["preds"].append(pred)
        aggregator[uid]["tgt"].append(tgt)

    labels, preds, tgts = [], [], []
    for uid, d in aggregator.items():
        labels.append(uid)
        preds.append(majority_vote(d["preds"]))
        tgts.append(majority_vote(d["tgt"]))

    pd.DataFrame({
        "Label": labels,
        "AvgPredicted": preds,
        "AvgTrue": tgts
    }).to_csv(f"predictions_bin_final_avg.csv",
                index=False)

### Learning curves &mdash; one per fold

Each subplot below shows train (blue) and validation (orange) loss per epoch for one of the `NUM_REPEATS x KFOLD = 15` fold-models. Curves that diverge late in training are the ones that benefited from early stopping.


In [ ]:
fig, axes = plt.subplots(
    NUM_REPEATS, KFOLD,
    figsize=(KFOLD * 2.6, NUM_REPEATS * 2.2),
    sharex=True, sharey=True,
)
# Make axes 2D even if NUM_REPEATS or KFOLD is 1
axes = np.atleast_2d(axes)

for h in all_loss_histories:
    ax = axes[h["rep"]][h["fold"]]
    ax.plot(h["epoch"], h["train"], color="tab:blue",   linewidth=1, label="train")
    ax.plot(h["epoch"], h["val"],   color="tab:orange", linewidth=1, label="val")
    ax.set_title(f"rep {h['rep']}  fold {h['fold']}", fontsize=9)
    ax.tick_params(labelsize=8)

# one shared legend in the top-left subplot
axes[0][0].legend(fontsize=8, loc="upper right")

# axis labels only on outer edges to reduce clutter
for r in range(NUM_REPEATS):
    axes[r][0].set_ylabel("BCE loss", fontsize=9)
for c in range(KFOLD):
    axes[-1][c].set_xlabel("epoch", fontsize=9)

fig.suptitle("Per-fold learning curves (BCEWithLogitsLoss)", y=1.02, fontsize=12)
fig.tight_layout()
plt.show()


## 10. Evaluation on the held-out test set

We load each of the 15 saved checkpoints and run it on `bin_tst_final.csv`. Predictions from all 15 models on every test row are concatenated into `predictions_test_bin.csv` (15&times; the test sequence count). Section 11 then majority-votes them down to one prediction per sequence.


In [ ]:
test_csv = os.path.join(
    data_dir,
    f"bin_tst_final.csv"
)

X_tst, y_tst, ids_tst = load_csv_data(test_csv, id_col, label_col)

predictions, metrics = [], []

for repeat_idx in range(NUM_REPEATS):
    for fold_idx in range(KFOLD):
        model_path = os.path.join(
            model_dir,
            f"nn_fold_{repeat_idx}_{fold_idx}_bin.pth"
        )


        net = Net(
            input_dim=X_tst.shape[1],     # use the TEST tensor's feature count
            model_type="bin",
            num_classes=1,                # ignored for binary
            hidden_size=HIDDEN_SIZE,
            hidden_layers=HIDDEN_LAYERS,
            dropout_input_output=DROPOUT_INPUT_OUTPUT,
            dropout_hidden=DROPOUT_HIDDEN,
        )

        net.load_state_dict(T.load(model_path, map_location="cpu"))

        mse,r2,pear,mcc,acc,fold_preds = evaluate_model(
            net, X_tst, y_tst, ids_tst
        )
        predictions.extend(fold_preds)
        metrics.append(
            {"MSE": mse, "R2": r2, "Pear": pear, "MCC": mcc, "Accuracy": acc}
        )


# ----------- save row-level predictions ------------------------- #
save_predictions(
    predictions,
    f"predictions_test_bin.csv"
)

# ----------- aggregate & save test metrics ---------------------- #
if metrics:
    avg_metrics = {}
    for key in metrics[0]:
        vals = [m[key] for m in metrics if not np.isnan(m[key])]
        avg_metrics[key] = float(np.mean(vals)) if vals else float('nan')


    with open(f"final_metrics_bin_tst.csv",
                "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["MSE","R2","Pear","MCC","Accuracy"])
        w.writeheader()
        w.writerow({k: fmt_float(v) for k,v in avg_metrics.items()})


## 11. Aggregate test predictions per sequence

`predictions_test_bin.csv` from section 10 is the *row-level* concatenation across all 15 checkpoints &mdash; every test sequence appears 15 times, once per `(rep, fold)`. We majority-vote those 15 calls into a single prediction per sequence, exactly mirroring what the training side already wrote to `predictions_bin_final_avg.csv`. The two `_final_avg.csv` files are the inputs to the confusion matrices in section 12.


In [ ]:
# Aggregate the row-level test predictions into one prediction per sequence.
#
# `predictions_test_bin.csv` (written by the loop above) is the *concatenation*
# of every (rep, fold) checkpoint's prediction on every test row, so each
# sequence appears NUM_REPEATS * KFOLD = 15 times. To get a single ensemble
# prediction per sequence we majority-vote those 15 calls. This mirrors what
# the training side does to produce `predictions_bin_final_avg.csv`, and is
# what downstream confusion-matrix cells should consume — otherwise the CM
# is computed on 15x the actual number of test sequences.

df_test_rows = pd.read_csv("predictions_test_bin.csv")

agg_test = defaultdict(lambda: {"preds": [], "tgt": []})
for _, row in df_test_rows.iterrows():
    agg_test[row["Label"]]["preds"].append(int(row["Predicted"]))
    agg_test[row["Label"]]["tgt"].append(int(row["True"]))

labels, preds, tgts = [], [], []
for uid, d in agg_test.items():
    labels.append(uid)
    preds.append(majority_vote(d["preds"]))
    tgts.append(majority_vote(d["tgt"]))

pd.DataFrame({
    "Label": labels,
    "AvgPredicted": preds,
    "AvgTrue": tgts,
}).to_csv("predictions_test_bin_final_avg.csv", index=False)

print(f"Aggregated {len(df_test_rows)} row-level predictions "
      f"into {len(labels)} per-sequence predictions.")


In [ ]:
# Preview the two aggregated prediction tables that the confusion matrices below consume.
print("Train (predictions_bin_final_avg.csv):")
print(pd.read_csv("predictions_bin_final_avg.csv").head().to_string())
print()
print("Test  (predictions_test_bin_final_avg.csv):")
print(pd.read_csv("predictions_test_bin_final_avg.csv").head().to_string())


## 12. Results: confusion matrices

Two views per split (train / test):

1. **Counts** &mdash; sklearn's `ConfusionMatrixDisplay` with raw counts. Quick to see where the model is right and where it's wrong.
2. **Row-percent** &mdash; each row sums to 100%, i.e. the *conditional accuracy per true class*. Useful when classes are imbalanced because a 90% global accuracy can hide a 0% recall on the minority class.

Counts and percentages are computed on the *per-sequence aggregated* predictions written in sections 9 and 11.


In [ ]:
# Counts view: train (left) and test (right) side-by-side.
df_train_preds = pd.read_csv("predictions_bin_final_avg.csv")
df_test_preds  = pd.read_csv("predictions_test_bin_final_avg.csv")

cm_train = confusion_matrix(df_train_preds["AvgTrue"].astype(int),
                            df_train_preds["AvgPredicted"].astype(int))
cm_test  = confusion_matrix(df_test_preds["AvgTrue"].astype(int),
                            df_test_preds["AvgPredicted"].astype(int))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ConfusionMatrixDisplay(cm_train, display_labels=["Non-binding (0)", "Binding (1)"]).plot(
    ax=axes[0], cmap="Blues", values_format='d', colorbar=False
)
axes[0].set_title("Train (per-sequence aggregated)")

ConfusionMatrixDisplay(cm_test, display_labels=["Non-binding (0)", "Binding (1)"]).plot(
    ax=axes[1], cmap="Purples", values_format='d', colorbar=False
)
axes[1].set_title("Test (per-sequence aggregated)")

fig.suptitle("NN binary classifier -- counts", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
def plot_confusion_matrix_row_percent(y_true, y_pred, title="Confusion Matrix", ax=None, cmap="Greens"):
    """
    Confusion matrix where EACH ROW sums to 100% (conditional accuracy per true class).
    If `ax` is given, draw into it; otherwise build a fresh figure.
    """
    cm = confusion_matrix(y_true, y_pred)
    cm_row_sum = cm.sum(axis=1, keepdims=True)
    cm_perc = np.where(cm_row_sum > 0, cm / cm_row_sum * 100, 0.0)

    # Class 0 = Non-binding, class 1 = Binding -- sklearn's confusion_matrix
    # returns rows/cols in ascending class-index order.
    labels = ["Non-binding", "Binding"]

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(cm_perc, cmap=cmap, vmin=0, vmax=100)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.set_xticks(np.arange(2))
    ax.set_yticks(np.arange(2))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]}\n({cm_perc[i,j]:.1f}%)",
                    ha='center', va='center', color='black', fontsize=12)

    if own_fig:
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.show()
    return im


In [ ]:
# Row-percent view: each row sums to 100% -> conditional accuracy per true class.
df_train_preds = pd.read_csv("predictions_bin_final_avg.csv")
df_test_preds  = pd.read_csv("predictions_test_bin_final_avg.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im_l = plot_confusion_matrix_row_percent(
    df_train_preds["AvgTrue"], df_train_preds["AvgPredicted"],
    title="NN (Train)", ax=axes[0], cmap="Greens",
)
im_r = plot_confusion_matrix_row_percent(
    df_test_preds["AvgTrue"], df_test_preds["AvgPredicted"],
    title="NN (Test)", ax=axes[1], cmap="Purples",
)

# one shared colourbar on the right
fig.colorbar(im_r, ax=axes, shrink=0.85, label="% of true class")
fig.suptitle("NN binary classifier -- row-percent (recall per true class)", y=1.04, fontsize=13)
plt.show()


## 13. Headline metrics summary

Mean +/- standard deviation across the 15 fold-models for the two metrics the canonical `run_model.py` reports (Accuracy and MCC), broken out by split. The Train row comes from per-fold validation metrics collected during training (`all_metrics`); the Test row comes from each checkpoint's evaluation on the held-out test set (`metrics`).

This is the format the paper uses to report headline binary numbers.


In [ ]:
def _mean_std(metric_list, key):
    """Mean and population std of `key` across a list of per-fold metric dicts, ignoring NaNs."""
    vals = [m[key] for m in metric_list
            if not (isinstance(m[key], float) and np.isnan(m[key]))]
    if not vals:
        return float("nan"), float("nan")
    return float(np.mean(vals)), float(np.std(vals))

rows = []
for split_name, ms in [("Train", all_metrics), ("Test", metrics)]:
    for metric_key, metric_label in [("Accuracy", "ACC"), ("MCC", "MCC")]:
        mu, sd = _mean_std(ms, metric_key)
        rows.append({
            "Model":  "NN (binary)",
            "Split":  split_name,
            "Metric": metric_label,
            "Mean":   mu,
            "Std":    sd,
            "Mean +/- Std": f"{mu:.3f} +/- {sd:.3f}",
        })

summary_df = pd.DataFrame(rows)

# Wide view: rows = (Model, Split), cols = ACC / MCC
display_df = summary_df.pivot(index=["Model", "Split"], columns="Metric", values="Mean +/- Std")
print("Headline metrics (mean +/- std across NUM_REPEATS x KFOLD folds):\n")
print(display_df.to_string())

# Also save to disk so process_results.ipynb-style downstream tools can read it
summary_df.to_csv("summary_metrics_bin.csv", index=False)
print("\nWrote summary_metrics_bin.csv")


## 14. Optional: scramble-fraction sweep

The headline result of the paper's binary section isn't just "the model gets X% accuracy" &mdash; it's that **as you progressively scramble the training labels, the model's performance degrades smoothly toward chance**. This is the strongest evidence that the model is learning real structure in the MMGBSA energies, not memorising sequence-specific noise.

The canonical pipeline runs this experiment via `run_ML.sh reg|bin|mclass "0.0 0.25 1.0"`, which re-does the entire data-prep + training pipeline at each fraction. We reproduce the binary version here in-memory.

**Cost:** With 5 fractions x 3 repeats x 5 folds = 75 NN trainings, this takes ~5-10 minutes on CPU. It's gated behind a flag so the rest of the notebook stays quick to run.


In [ ]:
# ----------------------------------------------------------------------------
# Set this to True to run the sweep. False keeps the notebook quick.
# ----------------------------------------------------------------------------
RUN_SCRAMBLE_SWEEP = False

SCRAMBLE_FRACTIONS = [0.0, 0.25, 0.5, 0.75, 1.0]


def scramble_sequences(df, id_col, label_col, frac, seed):
    """
    Reassign the label of `frac` of unique sequences to the label of a
    *different* randomly-chosen sequence. Ported from
    torch_prep_kfold.py::scramble_sequences in the canonical pipeline.
    """
    if frac <= 0.0:
        return df.copy()

    df_out = df.copy()
    unique_seqs = df_out[id_col].unique()
    n_to_scramble = int(len(unique_seqs) * frac)
    if n_to_scramble < 1:
        return df_out

    rng = np.random.default_rng(seed)
    scramble_seqs = rng.choice(unique_seqs, size=n_to_scramble, replace=False)
    for seq in scramble_seqs:
        possible = [s for s in unique_seqs if s != seq]
        src = rng.choice(possible)
        src_label = df_out.loc[df_out[id_col] == src, label_col].iloc[0]
        df_out.loc[df_out[id_col] == seq, label_col] = src_label
    return df_out


def run_one_scr_frac(df_train_in, df_test_in, scr, seed):
    """
    Re-run the entire prep + train + eval pipeline at a single scramble fraction.
    Returns (train_metrics_list, test_metrics_list) -- one entry per (rep, fold),
    each a dict with keys 'ACC' and 'MCC'.
    """
    # 1. Scramble labels on TRAIN only
    df_trn = scramble_sequences(df_train_in, id_col, label_col, scr, seed=seed)

    # 2. Re-run the train-prep chain (sections 5.1 - 5.3)
    df_trn = keep_last_n_percent(df_trn, id_col, KEEP_LAST_PERCENT)
    df_trn = average_features_for_mutants(df_trn, NAVG, id_col, label_col, seed)
    mean_df, std_df = compute_mean_std(df_trn, 'bin', id_col, label_col)
    stats_df = pd.merge(mean_df, std_df, on='colname')
    df_trn = apply_standardization(df_trn, stats_df, 'bin', id_col, label_col)
    df_trn = df_trn.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # 3. Apply the SAME train stats to test
    df_tst = keep_last_n_percent(df_test_in, id_col, KEEP_LAST_PERCENT)
    df_tst = average_features_for_mutants(df_tst, NAVG, id_col, label_col, seed)
    df_tst = apply_standardization(df_tst, stats_df, 'bin', id_col, label_col)

    feature_cols = [c for c in df_trn.columns if c not in [id_col, label_col]]
    X_tst = T.tensor(df_tst[feature_cols].values.astype(np.float32))
    y_tst = T.tensor(df_tst[label_col].values.astype(np.float32)).unsqueeze(1)
    ids_tst = df_tst[id_col].tolist()

    # 4. Repeated K-fold on the scrambled train set
    train_metrics, test_metrics = [], []
    for rep in range(NUM_REPEATS):
        kf = StratifiedGroupKFold(n_splits=KFOLD, shuffle=True, random_state=seed + 100 * rep)
        splits = kf.split(df_trn, df_trn[label_col], df_trn[id_col])
        for fold, (trn_idx, val_idx) in enumerate(splits):
            df_fold_trn = df_trn.iloc[trn_idx]
            df_fold_val = df_trn.iloc[val_idx]

            X_trn = T.tensor(df_fold_trn[feature_cols].values.astype(np.float32))
            y_trn = T.tensor(df_fold_trn[label_col].values.astype(np.float32)).unsqueeze(1)
            X_val = T.tensor(df_fold_val[feature_cols].values.astype(np.float32))
            y_val = T.tensor(df_fold_val[label_col].values.astype(np.float32)).unsqueeze(1)

            net = Net(
                input_dim=X_trn.shape[1],
                hidden_size=HIDDEN_SIZE,
                num_classes=1,
                hidden_layers=HIDDEN_LAYERS,
                dropout_input_output=DROPOUT_INPUT_OUTPUT,
                dropout_hidden=DROPOUT_HIDDEN,
                model_type='bin',
            )
            train_model(
                net, X_trn, y_trn, X_val, y_val,
                lrn_rate=LEARNING_RATE, wt_decay=WEIGHT_DECAY,
                max_epochs=MAX_EPOCHS,
                use_early_stopping=True, patience=PATIENCE,
                model_type='bin',
            )

            _, _, _, mcc_v, acc_v, _ = evaluate_model(net, X_val, y_val, df_fold_val[id_col].tolist())
            train_metrics.append({"ACC": acc_v, "MCC": mcc_v})

            _, _, _, mcc_t, acc_t, _ = evaluate_model(net, X_tst, y_tst, ids_tst)
            test_metrics.append({"ACC": acc_t, "MCC": mcc_t})

    return train_metrics, test_metrics


In [ ]:
if RUN_SCRAMBLE_SWEEP:
    sweep_rows = []
    for scr in SCRAMBLE_FRACTIONS:
        print(f"\n=== Scramble fraction = {scr} ===", flush=True)
        trn_ms, tst_ms = run_one_scr_frac(df_train, df_test, scr, seed=random_state)

        for split_name, ms in [("Train", trn_ms), ("Test", tst_ms)]:
            for metric_key, metric_label in [("ACC", "ACC"), ("MCC", "MCC")]:
                vals = [m[metric_key] for m in ms
                        if not (isinstance(m[metric_key], float) and np.isnan(m[metric_key]))]
                mu = float(np.mean(vals)) if vals else float("nan")
                sd = float(np.std(vals))  if len(vals) > 1 else 0.0
                sweep_rows.append({
                    "scr_frac": scr,
                    "split":    split_name,
                    "metric":   metric_label,
                    "mean":     mu,
                    "std":      sd,
                })
        print(f"  done ({len(trn_ms)} folds each for train/test).", flush=True)

    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv("sweep_metrics_bin.csv", index=False)
    print("\nWrote sweep_metrics_bin.csv\n")
    print(sweep_df.pivot_table(
        index=["split", "metric"], columns="scr_frac",
        values="mean").round(3).to_string())
else:
    print("Set RUN_SCRAMBLE_SWEEP = True (cell above) to run the sweep.")
    sweep_df = None


In [ ]:
if RUN_SCRAMBLE_SWEEP and sweep_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, metric in zip(axes, ["ACC", "MCC"]):
        for split, fmt, color in [("Train", "o-", "tab:blue"), ("Test", "s--", "tab:orange")]:
            sub = (sweep_df[(sweep_df["metric"] == metric) & (sweep_df["split"] == split)]
                   .sort_values("scr_frac"))
            ax.errorbar(sub["scr_frac"], sub["mean"], yerr=sub["std"],
                        fmt=fmt, capsize=4, color=color, label=split)
        ax.set_xlabel("Scramble fraction (training labels)")
        ax.set_ylabel(metric)
        ax.set_title(f"NN binary classifier: {metric} vs label-scrambling")
        ax.grid(alpha=0.3)
        ax.legend()
        # MCC's natural range includes negatives -- ACC's doesn't
        if metric == "ACC":
            ax.set_ylim(0.0, 1.05)
        else:
            ax.set_ylim(-0.2, 1.05)
    fig.tight_layout()
    plt.show()
else:
    print("(plot skipped -- RUN_SCRAMBLE_SWEEP is False)")
